已连接到 himga (Python 3.10.18)

# HiMGA-RAG基座使用指南

In [1]:
# Imports
import json
import pickle as pkl
from pathlib import Path

from himga.agent import BaseAgent
from himga.data import load_dataset
from himga.eval import compute_metrics, run_eval
from himga.eval.judge import batch_judge
from himga.llm import get_client
from himga.memory import NullMemory

/Users/zgh/Desktop/workingdir/HiMGA/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 加载数据集

In [2]:
# Load datasets
locomo_samples = load_dataset("locomo")
lme_samples = load_dataset("longmemeval", limit=10)

print(f"LoCoMo: {len(locomo_samples)} samples")
print(f"LongMemEval: {len(lme_samples)} samples")

0424-19:16|INFO| [locomo] found at /Volumes/itgz/datasets/locomo
0424-19:16|SUCCESS| Dataset 'locomo' loaded! (10 samples)
0424-19:16|INFO| [longmemeval] found at /Volumes/itgz/datasets/longmemeval
0424-19:16|SUCCESS| Dataset 'longmemeval' loaded! (10 samples)


LoCoMo: 10 samples
LongMemEval: 10 samples


## Inspect 数据集

In [12]:
# Inspect sessions of a single LoCoMo sample
sample_idx = 0
ses_idx = 0
msg_idx = 2

print(f"Sample {sample_idx}, Session {ses_idx}, Message {msg_idx}")
msg = locomo_samples[sample_idx].sessions[ses_idx].messages[msg_idx]
print(f"Session ID: {msg.session_id}")
print(f"Session Date: {msg.session_date}")
print(f"Turn ID: {msg.turn_id}")
print(f"Role: {msg.role}")
print(f"Content: {msg.content}")

Sample 0, Session 0, Message 2
Session ID: 1
Session Date: 2023-05-08 13:56:00
Turn ID: D1:3
Role: Caroline
Content: I went to a LGBTQ support group yesterday and it was so powerful.


In [4]:
# Inspect QA pairs of the same sample
qa_idx = 0
qa = locomo_samples[sample_idx].qa_pairs[qa_idx]
print(f"QAPair {qa_idx}: type={qa.question_type}")
print(f"Question: {qa.question}")
print(f"Evidence: {qa.evidence}")
print(f"Answer: {qa.answer}")

QAPair 0: type=temporal
Question: When did Caroline go to the LGBTQ support group?
Evidence: EvidenceRef(turn_ids=['D1:3'], session_ids=[])
Answer: 7 May 2023


## 建立Memory
himga内置无记忆Memory，`NullMemeory`

作为开发者，主要任务是建立自己的Memory类
Memory类需要实现以下接口：
- `ingest(message:Message)`: 将一个Message中的信息存储到Memory中
- `retrieve(query: str) -> str`: 根据一个查询字符串，从Memory中检索出证据
- `reset()`: 重置Memory,因为benchmark是多轮的，每个轮次需要重置记忆


In [17]:
# Smoke-test NullMemory ingest & retrieve
mem = NullMemory()
mem.ingest(
    message=locomo_samples[sample_idx].sessions[0].messages[0],
)
ans = mem.retrieve(query="What did the user say?")
print(f"Retrieved answer: {ans}")  # NullMemory should return empty string

Retrieved answer: 


## 建立Agent

这是HiMGA的自动测评智能体，给他`llm`客户端与`Memory`，他就就可以适配各类评估任务了

In [9]:
# Build agent
llm = get_client(provider="openai", model="gpt-4o-mini", batch_size=5)
agent = BaseAgent(memory=mem, llm=llm)
agent.answer(question=qa.question)

'Caroline went to the LGBTQ support group on March 15, 2023.'

## 回答数据集QAPair
使用如下：
`run_eval(samples, agent)`
便会自动`ingest`数据集中的每个Message，并根据QAPair中的问题进行检索，并在最后给出生成的回答，即如下的`results`

> 这里为了不浪费api 使用假数据

In [13]:
# Run evaluation (cached)
_cache = Path("outputs/locomo_results.pkl")
if _cache.exists():
    with open(_cache, "rb") as f:
        results = pkl.load(f)
else:
    results = run_eval([locomo_samples[0]], agent, show_progress=True)
    with open(_cache, "wb") as f:
        pkl.dump(results, f)

## 指标计算

In [14]:
# LLM-based judging (cached)
llm_scores = batch_judge(
    results,
    llm=llm,
    cache_path=Path("outputs/locomo_judge.json"),
)

In [15]:
# Compute & save metrics
metrics = compute_metrics(
    results,
    metrics=(
        "judge_score",  # -> 只有他要钱
        "exact_match",
        "f1",
        "rouge1",
        "rouge2",
        "rougeL",
        "bleu1",
        "bleu2",
        "bleu4",
        "meteor",
        "bert_f1",
        "sbert_similarity",
    ),
    llm=llm,  # 计算上述judge_score
    cache_path=Path("outputs/locomo_judge.json"),  # cache上述judge_score
)
with open("outputs/locomo_metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 6562.64it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6457.69it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                   

In [22]:
# Report metrics as DataFrame
def report_metrics(metrics):
    import pandas as pd

    rows = {"overall": metrics["overall"]}
    rows.update(metrics["by_type"])
    df = pd.DataFrame(rows)
    return df


report_metrics(metrics)

,overall,temporal,multi_hop,single_hop,open_domain,adversarial
judge_score,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
exact_match,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
f1,0.089080,0.072202,0.043111,0.033047,0.111057,0.120500
rouge1,0.098035,0.071347,0.057501,0.037813,0.128565,0.125787
rouge2,0.016014,0.000000,0.013002,0.003110,0.025140,0.024649
rougeL,0.087278,0.071347,0.045945,0.033714,0.110462,0.113190
bleu1,0.062554,0.045082,0.040124,0.038091,0.080745,0.072076
bleu2,0.023778,0.012603,0.016443,0.011619,0.032296,0.030195
bleu4,0.009460,0.007239,0.005136,0.005094,0.011676,0.012076
meteor,0.089977,0.015861,0.066298,0.029035,0.131114,0.135096
